In [0]:
############# BRONZE LAYER ###############
# Read the csv file
catalog = "workspace"
schema = "default"
volume = "my_volume"
filename = "credit_risk_dataset.csv"
filepath = "/Volumes/" + catalog + "/" + schema + "/" + volume
bronze_table_name = "bronze_credit_risk"
from pyspark.sql.types import *
from pyspark.sql.functions import *
df = (spark.read
      .option("header", True)
      .option("inferSchema", True)
      .option("multiline", True)
      .csv(f"{filepath}/{filename}")
)

# Save the raw data
df.write.mode("overwrite").saveAsTable(bronze_table_name)

In [0]:
###### SILVER LAYER - DATA CLEANING #########################
silver_table_name = "silver_credit_risk"
silver_df = spark.table(bronze_table_name)

# Step 1: drop duplicates
silver_df = silver_df.dropDuplicates()
# Step 4: trim whitespaces in string columns
for field in silver_df.schema.fields:
    if isinstance(field.dataType, StringType):
        silver_df = silver_df.withColumn(field.name, trim(col(field.name)))
# Step 2: rename columns for reading
silver_df = silver_df.withColumnRenamed("person_home_ownership", "home_ownership")
silver_df = silver_df.withColumnRenamed("loan_amnt", "loan_amount")
silver_df = silver_df.withColumnRenamed("person_emp_length", "employment_length")
silver_df = silver_df.withColumnRenamed("loan_int_rate", "interest_rate")
silver_df = silver_df.withColumnRenamed("cb_person_default_on_file", "default_history")
silver_df = silver_df.withColumnRenamed("cb_person_cred_hist_length", "credit_history_length")
# Step 3: standardize categorical name
silver_df = silver_df.withColumn("home_ownership", regexp_replace(col("home_ownership"), "MORTGAGE", "mortgage"))
silver_df = silver_df.withColumn("home_ownership", regexp_replace(col("home_ownership"), "RENT", "rent"))
silver_df = silver_df.withColumn("home_ownership", regexp_replace(col("home_ownership"), "OWN", "own"))
silver_df = silver_df.withColumn("loan_intent", regexp_replace(col("loan_intent"), "DEBTCONSOLIDATION", "debt_consolidation"))
silver_df = silver_df.withColumn("loan_intent", regexp_replace(col("loan_intent"), "EDUCATION", "education"))
silver_df = silver_df.withColumn("loan_intent", regexp_replace(col("loan_intent"), "MEDICAL", "medical"))
silver_df = silver_df.withColumn("loan_intent", regexp_replace(col("loan_intent"), "PERSONAL", "personal"))
silver_df = silver_df.withColumn("loan_intent", regexp_replace(col("loan_intent"), "VENTURE", "venture"))
silver_df = silver_df.withColumn("loan_intent", regexp_replace(col("loan_intent"), "HOMEIMPROVEMENT", "homeimprovement"))
silver_df = silver_df.withColumn("loan_intent", regexp_replace(col("loan_intent"), "WEDDING", "wedding"))
# Step 5: convert Y/N in previous default history to boolean
silver_df = silver_df.withColumn("default_history", regexp_replace(col("default_history"), "Y", "true"))
silver_df = silver_df.withColumn("default_history", regexp_replace(col("default_history"), "N", "false"))
silver_df.write.mode("overwrite").saveAsTable(silver_table_name)   
display(silver_df)                             

person_age,person_income,home_ownership,employment_length,loan_intent,loan_grade,loan_amount,interest_rate,loan_status,loan_percent_income,default_history,credit_history_length
22,59000,rent,123.0,personal,D,35000,16.02,1,0.59,true,3
21,9600,own,5.0,education,B,1000,11.14,0,0.1,false,2
25,9600,mortgage,1.0,medical,C,5500,12.87,1,0.57,false,3
23,65500,rent,4.0,medical,C,35000,15.23,1,0.53,false,2
24,54400,rent,8.0,medical,C,35000,14.27,1,0.55,true,4
21,9900,own,2.0,venture,A,2500,7.14,1,0.25,false,2
26,77100,rent,8.0,education,B,35000,12.42,1,0.45,false,3
24,78956,rent,5.0,medical,B,35000,11.11,1,0.44,false,4
24,83000,rent,8.0,personal,A,35000,8.9,1,0.42,false,2
21,10000,own,6.0,venture,D,1600,14.74,1,0.16,false,3


In [0]:
########## GOLDEN LAYER ######################################
## Portfolio overview
df = spark.table("silver_credit_risk")
# compute total loans
total_loans = df.count()
# total loan amount
total_loan_amnt = df.select("loan_amount").agg(sum("loan_amount")).collect()[0][0]
# average loan amount
avg_loan_amnt = df.select("loan_amount").agg(round(avg("loan_amount"), 2)).collect()[0][0]
# average interest rate
avg_interest_rate = df.select("interest_rate").agg(round(avg("interest_rate"), 2)).collect()[0][0]
# default rate
default_rate = df.select("loan_status").agg(sum("loan_status")).collect()[0][0]/total_loans
# average borrower income
avg_borrower_income = df.select("person_income").agg(round(avg("person_income"), 2)).collect()[0][0]

# create a table with the previously computed quantities
kpi_df = spark.createDataFrame([
    Row(
        total_loans=total_loans,
        total_loan_amount=total_loan_amnt,
        average_loan_amount=avg_loan_amnt,
        average_interest_rate=avg_interest_rate,
        default_rate=default_rate,
        average_borrower_income=avg_borrower_income
    )
])

# Add column with percentages for visualization purposes
kpi_df = kpi_df.withColumn(
    "default_rate_pct",
    round(col("default_rate") * 100, 2)
)
kpi_df.display()


total_loans,total_loan_amount,average_loan_amount,average_interest_rate,default_rate,average_borrower_income,default_rate_pct
32416,310994100,9593.85,11.02,0.21868830207305034,66091.64,21.87


In [0]:
# Analyse metrics for the borrowers who defaulted and for those who didnt: avg age, avg income, avg interest rate, avg loan amnt
df = spark.table("silver_credit_risk")
default_vs_nodefault_clients = (df.withColumn("default_status",
                                              when(col("loan_status") == 1, "default")
                                              .otherwise("no_default")
                                              )
                                )
key_metrics = (
                default_vs_nodefault_clients.groupBy("default_status")
                .agg(round(avg("person_age"), 2).alias("avg_age"),
                     round(avg("person_income"), 3).alias("avg_income"),
                     round(avg("interest_rate"), 2).alias("avg_interest_rate"),
                     round(avg("loan_amount"), 3).alias("avg_loan_amnt"))
               )

key_metrics.display()

default_status,avg_age,avg_income,avg_interest_rate,avg_loan_amnt
default,27.47,49094.498,13.07,10857.48
no_default,27.82,70849.123,10.44,9240.156


In [0]:
## Default rate by loan purpose
# Compute default rate by loan purpose
df = spark.table("silver_credit_risk")
default_rate_by_purpose = (df.groupBy("loan_intent")
                            .agg(
                                    count("*").alias("total_loans"), 
                                    sum("loan_status").alias("total_defaults")
                                )
                            .withColumn("default_rate", col("total_defaults")/col("total_loans")).orderBy(col("default_rate").desc())
)
# Add column with percentages for visualization purposes
default_rate_by_purpose = default_rate_by_purpose.withColumn(
    "default_rate_by_purpose_pct",
    round(col("default_rate") * 100, 2)
)
default_rate_by_purpose.display()




loan_intent,total_loans,total_defaults,default_rate,default_rate_by_purpose_pct
debt_consolidation,5189,1488,0.28676045480824824,28.68
medical,6042,1617,0.2676266137040715,26.76
homeimprovement,3594,940,0.2615470228158041,26.15
personal,5498,1094,0.1989814477991997,19.9
education,6411,1106,0.17251598814537514,17.25
venture,5682,844,0.14853924674410418,14.85


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
## Compute default rate by loan grade
df = spark.table("silver_credit_risk")
default_rate_by_grade = (
                            df.groupBy("loan_grade").agg(
                                count("*").alias("total_loans"), 
                                sum("loan_status").alias("total_defaults")
                                )
                            .withColumn("default_rate", col("total_defaults")/col("total_loans")).orderBy(col("default_rate").desc())
                        )
# add a column with percentages, for visualization purposes
default_rate_by_grade = default_rate_by_grade.withColumn("default_rate_by_grade_pct", round(col("default_rate") * 100, 2))

default_rate_by_grade.display()

loan_grade,total_loans,total_defaults,default_rate,default_rate_by_grade_pct
G,64,63,0.984375,98.44
F,241,170,0.7053941908713693,70.54
E,963,621,0.6448598130841121,64.49
D,3620,2138,0.5906077348066299,59.06
C,6438,1336,0.20751786269027647,20.75
B,10387,1695,0.16318475016847983,16.32
A,10703,1066,0.09959824348313558,9.96


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
## Default rate by age group
df = spark.table("silver_credit_risk")
# group loaners by age group: 18-25, 26-35, 36-45, 46-55, 56+
age_groups = (df.withColumn(
                            "age_group",
                             when(col("person_age")<=25, "18-25")
                             .when(col("person_age")<=35, "26-35")
                             .when(col("person_age")<= 45, "36-45")
                             .when(col("person_age")<=55, "45-55")
                             .otherwise("56+")
                            )
)
# compute default rate by age group
default_rate_by_age = (
    age_groups.groupBy("age_group").agg(count("*").alias("total_loans"),
                                    sum("loan_status").alias("total_defaults")
                                    )
                                    .withColumn("default_rate", col("total_defaults")/col("total_loans"))
                                    .orderBy("age_group")
)
default_rate_by_age = default_rate_by_age.withColumn(
    "default_rate_pct",
        round(col("default_rate") * 100, 2)
)
display(default_rate_by_age)

age_group,total_loans,total_defaults,default_rate,default_rate_pct
18-25,15245,3527,0.23135454247294196,23.14
26-35,13711,2837,0.20691415651666545,20.69
36-45,2809,581,0.20683517265930937,20.68
45-55,512,112,0.21875,21.88
56+,139,32,0.2302158273381295,23.02


In [0]:
df = spark.table("silver_credit_risk")

# Compute default rates by borrower income group: <30K, 30K-60K, 60K-100K, >100K
income_groups = (df.withColumn(
                                "yearly_income_group",
                                when(col("person_income") <= 30000, "<30K")
                                .when(col("person_income") <= 60000, "30K-60K")
                                .when(col("person_income") <= 100000, "60K-100K")
                                .otherwise(">100K")
                            )
                 )

default_rate_by_income = (income_groups.groupBy("yearly_income_group")
                          .agg(count("*").alias("total_loans"),
                               sum("loan_status").alias("total_defaults"),
                                round(avg(col("person_age")), 2).alias("average_age"),
                                round(avg(col("loan_amount")), 3).alias("average_loan_amount")
                               )
                          .withColumn("default_rate", col("total_defaults")/col("total_loans"))
                          .orderBy("default_rate")
                        )
default_rate_by_income = default_rate_by_income.withColumn("default_rate_pct", round(col("default_rate")*100, 2))
default_rate_by_income.display()

yearly_income_group,total_loans,total_defaults,average_age,average_loan_amount,default_rate,default_rate_pct
>100K,4191,400,29.38,14267.478,0.09544261512765449,9.54
60K-100K,9603,1285,27.96,11203.939,0.13381235030719565,13.38
30K-60K,14136,3359,27.37,8428.279,0.23762026032823996,23.76
<30K,4486,2045,26.96,5453.751,0.45586268390548373,45.59


Databricks visualization. Run in Databricks to view.

# **Key Findings**
- Borrowers aged **18-25** exhibit the highest default rate (23.14%), followed by borrowers aged **56+** (23.02%);
- Loan grades **F-G** show substantial default rate, while loan grades **A-B** show low default rate;
- Lower income clients (**<30K** yearly) show susbtantially higher default rates;
- Loans for **medical** purposes, **home improvement** and **debt consolidation** show the highest default rate;
- High risk clients borrowed on average 5.45 thousand dollars, and the average age of high risk clients is 26.9.
- The bank gave, in total, 4486 loans to high risk clients.